#Dataset

##Utils

In [1]:
import warnings
warnings.filterwarnings("ignore")

### Dataset Verification

In [40]:
###################################################
# script for verifying the dataset
# format of input data
#   - images (.jpg, .png ....)
#   - labels (yolo_format_annotations(cls, x_cen, y_cen, width, height)) # normalized values
#####################################################
import os
import cv2
import random
from google.colab.patches import cv2_imshow


def xywh_to_xyxy(x_center, y_center,
                 bbox_width, bbox_height,
                 img_width, img_height):
  """Converting normalized yolo labels (class_id, x, y, w, h) into unnormalized (class_id, x1, y1, x2, y2)

    Return:
      list (class_id, x1, y1, x2, y2)
  """

  # Denormalize YOLO coords
  x_center *= img_width
  y_center *= img_height
  bbox_width *= img_width
  bbox_height *= img_height

  x1 = int(x_center - bbox_width / 2)
  y1 = int(y_center - bbox_height / 2)
  x2 = int(x_center + bbox_width / 2)
  y2 = int(y_center + bbox_height / 2)

  return [x1, y1, x2, y2]


def draw_bbox(image, lines, line_width=2):
  """Draw bounding box over the image

    Return:
      Image array
  """
  h, w = image.shape[:2]

  for idx_line, line in enumerate(lines):

    parts = line.strip().split()
    if len(parts) != 5:
        continue

    class_id, x_center, y_center, bbox_width, bbox_height = map(float, parts)
    class_id, x1, y1, x2, y2 = int(class_id), *xywh_to_xyxy(x_center, y_center,
                                                            bbox_width, bbox_height,
                                                            w, h)

    # Pick color for this class
    color = (random.randint(0, 255),
            random.randint(0, 255),
            random.randint(0, 255))
    global colors # defined in __name__=="__main__" memory block

    if class_id not in colors.keys():

      while True:
        # prevent duplicate colors for multiple classes
        if color in colors.values():
            color = (random.randint(0, 255),
                    random.randint(0, 255),
                    random.randint(0, 255))
            continue

        else:
          # add new color for new class
          colors[class_id] = color
          break


    color = colors[class_id]

    # Draw rectangle + class ID
    cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
    cv2.putText(image, str(class_id), (x1, max(20, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

  return image


def verification_main(image_path, label_path, image_save_path):
  """Main Code looping images patha and labels path; saving resltand images

  """

  # read image
  image = cv2.imread(image_path)

  if image is None:
    print(f"Could not read image: {image_path}")
    return

  # read labels
  with open(label_path, 'r') as label_file:
    lines = label_file.readlines()

  image = draw_bbox(image, lines)

  # save image
  cv2.imwrite(image_save_path, image)
  print("Image {} Saved to {}".format(image_path, image_save_path))


### Generalizing Class Of Sub Batches

In [25]:
###################################################
# script for generalizing the class of the sub batches into global classes list
# format of input data
#   - images (.jpg, .png ....)
#   - labels (yolo_format_annotations(cls, x_cen, y_cen, width, height)) # normalized values
#
# RETURNS
# A new directory following the yolo structure
# creates a global symbols dict for combining all the classes considering the name of the class
#####################################################
import os
import cv2
import random
import shutil
import yaml
from google.colab.patches import cv2_imshow


def get_class(class_, flag='id'):
  """Extract Class Name against and ID and voice versa

  Args:
    class_ (str): Class_id if flag='id' else Class_name if flag='name'

  Return:
    class_id or class_name
  """
  global generalized_classes

  if flag == 'id': # return class_name

    if class_ not in list(generalized_classes.keys()):
      generalized_classes[len(generalized_classes)] = class_

    return generalized_classes[class_]


  elif flag == 'name': # return class_id

    if class_ not in list(generalized_classes.values()):
      generalized_classes[len(generalized_classes)] = class_

    return list(generalized_classes.keys())[list(generalized_classes.values()).index(class_)]

  else:
    raise ValueError("Flag must be 'id' or 'name'")
    return


def read_yaml_file(yaml_path):
  """Read yaml file and return the dict(Keys=class_id: Value=class_name)

    Return:
      dict
  """
  with open(yaml_path, 'r') as yaml_file:
    yaml_data = yaml.safe_load(yaml_file)

  return yaml_data['names']


def write_yaml_file(yaml_path, classes_list):
  """Write yaml file
  """
  dump_data = {

    'train': '../train/images',
    'nc': f"{len(classes_list)}",

    'names': list(classes_list.values())
  }

  with open(yaml_path, 'w') as file:
    yaml.dump(dump_data, file, sort_keys=False)

  print("Yaml file saved to {}".format(yaml_path))



def write_file(file_path, data):
  """Write data to file

  """
  with open(file_path, 'w') as file:
    for line in data:
      file.write(line + '\n')



def parse_labels(file_path, write_path, class_list):
  """Parse Yolo Data And Write Back

    Return:
  """
  # vars
  labels_ = []

  # read labels
  with open(file_path, 'r') as label_file:
    lines = label_file.readlines()

  for idx_line, line in enumerate(lines):
    parts = line.strip().split()

    if len(parts) != 5:
        continue

    class_id, x_center, y_center, bbox_width, bbox_height = map(float, parts)

    class_name = class_list[int(class_id)]

    class_id = get_class(class_name, flag='name') # return id

    labels_.append(f"{class_id} {x_center} {y_center} {bbox_width} {bbox_height}")


  # write file
  write_file(write_path, labels_)



def main(read_dir, write_dir, g_idx=0):
  """Main Code looping images patha and labels path; saving resltand images

  """

  # paths
  images_path = os.path.join(read_dir, "train", "images")
  labels_path = os.path.join(read_dir, "train", "labels")
  yaml_path = os.path.join(read_dir, "data.yaml")


  # write dataset
  images_save_path = os.path.join(write_dir, "train", "images")
  labels_save_path = os.path.join(write_dir, "train", "labels")
  yaml_path_save = os.path.join(write_dir, "data.yaml")

  os.makedirs(images_save_path, exist_ok=True)
  os.makedirs(labels_save_path, exist_ok=True)


  # read yaml file
  class_list = read_yaml_file(yaml_path)
  print(class_list)

  for img_idx, img_file in enumerate(os.listdir(images_path)):

    image_path = os.path.join(images_path, img_file)
    label_path = os.path.join(labels_path, img_file[:-3] + "txt")


    # copy image & write label txt
    save_label_path = os.path.join(labels_save_path, img_file[:-3] + "txt")
    shutil.copy(image_path, os.path.join(images_save_path, img_file))
    parse_labels(label_path, save_label_path, class_list)

    print("Image {} Copied to {} - label written to {}".format(image_path, os.path.join(images_save_path, img_file), save_label_path))


  global generalized_classes
  write_yaml_file(yaml_path_save, generalized_classes)

### Per Class Samples Analytics

In [ ]:
import os
import cv2
import yaml
import json


###################################################
# script for verifying the dataset
# format of input data
#   - images (.jpg, .png ....)
#   - labels (yolo_format_annotations(cls, x_cen, y_cen, width, height)) # normalized values
#####################################################
import os
import cv2
import random
from google.colab.patches import cv2_imshow


def get_class(class_, flag='id'):
  """Extract Class Name against and ID and voice versa

  Args:
    class_ (str): Class_id if flag='id' else Class_name if flag='name'

  Return:
    class_id or class_name
  """
  global generalized_classes

  if flag == 'id': # return class_name

    if class_ not in list(generalized_classes.keys()):
      generalized_classes[len(generalized_classes)] = class_

    return generalized_classes[class_]


  elif flag == 'name': # return class_id

    if class_ not in list(generalized_classes.values()):
      generalized_classes[len(generalized_classes)] = class_

    return list(generalized_classes.keys())[list(generalized_classes.values()).index(class_)]

  else:
    raise ValueError("Flag must be 'id' or 'name'")
    return



def read_yaml_file(yaml_path):
  """Read yaml file and return the dict(Keys=class_id: Value=class_name)

    Return:
      dict
  """
  with open(yaml_path, 'r') as yaml_file:
    yaml_data = yaml.safe_load(yaml_file)

  return yaml_data['names']



def parse_labels(image, lines, line_width=2):
  """Parse Yolo Data And Write Back

    Return:
  """
  h, w = image.shape[:2]

  for idx_line, line in enumerate(lines):

    parts = line.strip().split()
    if len(parts) != 5:
        continue

    class_id, x_center, y_center, bbox_width, bbox_height = map(float, parts)




def main(image_path, label_path, image_save_path):
  """Main Code looping images patha and labels path; saving resltand images

  """

  # read image
  image = cv2.imread(image_path)

  if image is None:
    print(f"Could not read image: {image_path}")
    return

  # read labels
  with open(label_path, 'r') as label_file:
    lines = label_file.readlines()

  image = draw_bbox(image, lines)

  # save image
  cv2.imwrite(image_save_path, image)
  print("Image {} Saved to {}".format(image_path, image_save_path))


## Unzip From Drive

In [20]:
from google.colab import drive
drive.mount("Ibrah")

Mounted at Ibrah


In [21]:
import os

zip_data_path = "/content/Ibrah/MyDrive/Symbol_Detection/Classification/Classification_Datasets/Roboflow Symbols Batch.zip"

!unzip -q {zip_data_path.replace(" ", "\ ")} -d /content/

##Unzip Sub Batches


In [22]:
import os

# -- dataset paths --
list_paths = ["/content/Roboflow Symbols Batch/First Batch.v8i.yolov8.zip", "/content/Roboflow Symbols Batch/Second_batch.v10i.yolov8.zip",
              "/content/Roboflow Symbols Batch/Third Batch Z1.v9i.yolov8.zip", "/content/Roboflow Symbols Batch/Third Batch Z2.v8i.yolov8.zip",
              "/content/Roboflow Symbols Batch/Third Batch.v8i.yolov8.zip", "/content/Roboflow Symbols Batch/batch-4-raw_annotated_dataset.v8i.yolov8.zip",
              "/content/Roboflow Symbols Batch/Batch 5 - raw_annotated_dataset.v8i.yolov8.zip", "/content/Roboflow Symbols Batch/Batch 6 - raw_annotated_dataset.v8i.yolov8.zip",
              "/content/Roboflow Symbols Batch/Batch 7- raw_annotated_dataset_05.v8i.yolov8.zip", "/content/Roboflow Symbols Batch/batch_8_raw_annotated_dataset.v7i.yolov8.zip"]

# -- save paths for each zip --
save_paths = ["dataset_01", "dataset_02", "dataset_03_01", "dataset_03_02",
              "dataset_03_03", "dataset_04", "dataset_05", "dataset_06",
              "dataset_07", "dataset_08"]


save_base_path = "/content/combine_symbol_v0"

os.makedirs(save_base_path, exist_ok=True)


for idx in range(len((list_paths))):
  print("Unziping {} to {}".format(list_paths[idx], os.path.join(save_base_path, save_paths[idx])))

  !unzip -q {list_paths[idx].replace(" ", "\ ")} -d {os.path.join(save_base_path, save_paths[idx])}

Unziping /content/Roboflow Symbols Batch/First Batch.v8i.yolov8.zip to /content/combine_symbol_v0/dataset_01
Unziping /content/Roboflow Symbols Batch/Second_batch.v10i.yolov8.zip to /content/combine_symbol_v0/dataset_02
Unziping /content/Roboflow Symbols Batch/Third Batch Z1.v9i.yolov8.zip to /content/combine_symbol_v0/dataset_03_01
Unziping /content/Roboflow Symbols Batch/Third Batch Z2.v8i.yolov8.zip to /content/combine_symbol_v0/dataset_03_02
Unziping /content/Roboflow Symbols Batch/Third Batch.v8i.yolov8.zip to /content/combine_symbol_v0/dataset_03_03
Unziping /content/Roboflow Symbols Batch/batch-4-raw_annotated_dataset.v8i.yolov8.zip to /content/combine_symbol_v0/dataset_04
Unziping /content/Roboflow Symbols Batch/Batch 5 - raw_annotated_dataset.v8i.yolov8.zip to /content/combine_symbol_v0/dataset_05
Unziping /content/Roboflow Symbols Batch/Batch 6 - raw_annotated_dataset.v8i.yolov8.zip to /content/combine_symbol_v0/dataset_06
Unziping /content/Roboflow Symbols Batch/Batch 7- raw

### Dataset Verification

In [43]:
###################################################
# REFERENCE: Functions defined in Dataset Utils
#
# script for verifying the dataset
# format of input data
#   - images (.jpg, .png ....)
#   - labels (yolo_format_annotations(cls, x_cen, y_cen, width, height)) # normalized values
#####################################################
import random

if __name__ == "__main__":
  colors = {}

  images_path = "/content/final_dataset/images"
  labels_path = "/content/final_dataset/labels"

  save_path = "/content/tests_saved"
  samples = 20 # number of samples to be used
  patch_size = 1792  # Not directly used unless you want to draw grid or check sizes

  os.makedirs(save_path, exist_ok=True)


  for img_idx, img_file in enumerate(random.choices(os.listdir(images_path), k=samples)):

    if not img_file.lower().endswith(('.jpg', '.png', '.jpeg')):
        continue

    image_path = os.path.join(images_path, img_file)
    label_path = os.path.join(labels_path, img_file.rsplit('.', 1)[0] + ".txt")

    image_save_path = os.path.join(save_path, img_file)

    verification_main(image_path, label_path, image_save_path)


    if img_idx >= samples:
      break

Image /content/final_dataset/images/295.jpg Saved to /content/tests_saved/295.jpg
Image /content/final_dataset/images/510.jpg Saved to /content/tests_saved/510.jpg
Image /content/final_dataset/images/472.jpg Saved to /content/tests_saved/472.jpg
Image /content/final_dataset/images/367.jpg Saved to /content/tests_saved/367.jpg
Image /content/final_dataset/images/195.jpg Saved to /content/tests_saved/195.jpg
Image /content/final_dataset/images/47.jpg Saved to /content/tests_saved/47.jpg
Image /content/final_dataset/images/181.jpg Saved to /content/tests_saved/181.jpg
Image /content/final_dataset/images/16.jpg Saved to /content/tests_saved/16.jpg
Image /content/final_dataset/images/378.jpg Saved to /content/tests_saved/378.jpg
Image /content/final_dataset/images/361.jpg Saved to /content/tests_saved/361.jpg
Image /content/final_dataset/images/714.jpg Saved to /content/tests_saved/714.jpg
Image /content/final_dataset/images/155.jpg Saved to /content/tests_saved/155.jpg
Image /content/final

In [42]:
!rm -rf /content/tests_saved

### Generalizing Class Of Sub Batches

In [27]:
if __name__=="__main__":

  # vars
  generalized_classes = {}


  # batches paths
  root_path = "/content/combine_symbol_v0"
  list_batches = os.listdir(root_path)


  # global dataset paths
  save_base_path = "/content/combine_symbol_v1"
  os.makedirs(save_base_path, exist_ok=True)


  for batch_idx, batch in enumerate(list_batches):
    print("Processing Batch {}/{}".format(batch_idx+1, len(list_batches)))

    # batch
    batch_path = os.path.join(root_path, batch)
    main(batch_path, save_base_path, g_idx=batch_idx)

Processing Batch 1/10
['0', '3-WAY SOLENOID VALVE', 'BATTERY LIMIT', 'Ball Valve', 'Blind Flange', 'Break Symbol', 'Butterfly Valve', 'Cap', 'Check Valve', 'Concentric', 'DCS', 'DCS Func Access in Aux Loc', 'DCS Single-Func Field Mounted', 'DCS Single-Func access in prime location', 'DESIGN PRESSURE LIMIT', 'Flange', 'Flanged Nozzle', 'Flow Direction', 'Flow Meter', 'Gate Valve', 'Globe Type Control Valve', 'Globe Valve', 'Interlock', 'MANHOLE', 'Motor', 'Motor Actuator With Valve', 'Needle Valve', 'Off-Drawing', 'Orifice Plate and Flanges With Meter', 'Piston Actuator With Valve', 'Plug', 'Pressure Relief Valve With Actuator', 'Pump', 'Ruptured Disc', 'Slope', 'Spectacle Blind_Close', 'Swing Check Valve', 'Tag with balloon', 'Tundish', 'Y Type Strainer', 'line split arrows', 'plc']
Image /content/combine_symbol_v0/dataset_03_02/train/images/bw_image_8_1_patch_2_2_jpg.rf.5d74cd76485d1f388badbd32a35c7d70.jpg Copied to /content/combine_symbol_v1/train/images/bw_image_8_1_patch_2_2_jpg.rf

In [28]:
!pip install -q roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 98.6 MB/s eta 0:00:00


In [ ]:
https://app.roboflow.com/institute-of-management-sciences-eacvm

In [44]:
from roboflow import Roboflow
from google.colab import userdata

rf = Roboflow(api_key=str(userdata.get("r_b_ibrah")))

workspace = rf.workspace("institute-of-management-sciences-eacvm")

workspace.upload_dataset(
    dataset_path="/content/final_dataset",
    project_name="symbol_dataset_fine_tune",
    num_workers=8
)

loading Roboflow workspace...
loading Roboflow project...
Uploading to existing project institute-of-management-sciences-eacvm/symbol_dataset_fine_tune
[UPLOADED] /content/final_dataset/images/3.jpg (0purZZqFdaRtk5MFdmY8) [1.2s] / annotations = OK [0.4s]
[UPLOADED] /content/final_dataset/images/2.jpg (VXHoaLUTzrabkYU2nXtZ) [1.3s] / annotations = OK [0.4s]
[UPLOADED] /content/final_dataset/images/0.jpg (4F5M6t6EUbVblLxflgtD) [1.4s] / annotations = OK [0.4s]
[UPLOADED] /content/final_dataset/images/5.jpg (GG1V43YKkzwlud9PiHxS) [1.4s] / annotations = OK [0.5s]
[UPLOADED] /content/final_dataset/images/4.jpg (0purZZqFdaRtk5MFdmY8) [1.3s] / annotations = OK [0.7s]
[UPLOADED] /content/final_dataset/images/1.jpg (GG1V43YKkzwlud9PiHxS) [1.7s] / annotations = OK [0.6s]
[UPLOADED] /content/final_dataset/images/8.jpg (7H0oKJsQSbcGJd1yq14q) [0.8s] / annotations = OK [0.3s]
[UPLOADED] /content/final_dataset/images/7.jpg (jB0vu61n44TJLZcdqom7) [1.5s] / annotations = OK [1.3s]
[UPLOADED] /content/fina

### Classes Analytic & Jsonification

In [36]:
!rm -rf /content/final_dataset

In [37]:
import os
import shutil
import random
import math

# =========================
# CONFIG
# =========================
INPUT_DATASET = "/content/combine_symbol_v1/train"
OUTPUT_DATASET = "/content/final_dataset"

MAX_IMAGES = 1000  # number of images to select

BETA = 0.7  # diversity weight

# Optional bucket thresholds
HIGH_THRESHOLD = 20
MEDIUM_THRESHOLD = 5

# =========================
# PATHS
# =========================
img_dir = os.path.join(INPUT_DATASET, "images")
lbl_dir = os.path.join(INPUT_DATASET, "labels")

# =========================
# FUNCTION: Extract stats
# =========================
def get_stats(label_path):
    """
    Returns:
        num_symbols (int)
        unique_classes (set)
    """
    if not os.path.exists(label_path):
        return 0, set()

    classes = set()
    count = 0

    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue

            cls_id = int(parts[0])
            classes.add(cls_id)
            count += 1

    return count, classes

# =========================
# STEP 1: COLLECT DATA
# =========================
dataset = []

for file in os.listdir(img_dir):
    if not file.lower().endswith((".jpg", ".png", ".jpeg")):
        continue

    img_path = os.path.join(img_dir, file)
    lbl_path = os.path.join(lbl_dir, file.rsplit(".", 1)[0] + ".txt")

    num_symbols, class_set = get_stats(lbl_path)
    num_unique = len(class_set)

    # Score = density + diversity
    score = math.log(1 + num_symbols) + BETA * num_unique

    dataset.append({
        "img": img_path,
        "lbl": lbl_path,
        "score": score,
        "num_symbols": num_symbols,
        "num_unique": num_unique
    })

print(f"Total images: {len(dataset)}")

# =========================
# STEP 2: SORT BY SCORE
# =========================
dataset = sorted(dataset, key=lambda x: x["score"], reverse=True)

# =========================
# STEP 3: (OPTIONAL) BUCKETING
# =========================
high, medium, low = [], [], []

for item in dataset:
    c = item["num_symbols"]

    if c > HIGH_THRESHOLD:
        high.append(item)
    elif c > MEDIUM_THRESHOLD:
        medium.append(item)
    else:
        low.append(item)

print(f"Buckets -> High: {len(high)}, Medium: {len(medium)}, Low: {len(low)}")

# =========================
# STEP 4: SELECT IMAGES
# =========================
selected = []

def sample_from_bucket(bucket, n):
    if len(bucket) == 0:
        return []
    return bucket[:min(n, len(bucket))]  # already sorted by score

# distribute selection
n_high = int(MAX_IMAGES * 0.4)
n_medium = int(MAX_IMAGES * 0.3)
n_low = int(MAX_IMAGES * 0.3)

selected += sample_from_bucket(high, n_high)
selected += sample_from_bucket(medium, n_medium)
selected += sample_from_bucket(low, n_low)

print(f"Selected images: {len(selected)}")

# =========================
# STEP 5: COPY DATA
# =========================
out_img_dir = os.path.join(OUTPUT_DATASET, "images")
out_lbl_dir = os.path.join(OUTPUT_DATASET, "labels")

os.makedirs(out_img_dir, exist_ok=True)
os.makedirs(out_lbl_dir, exist_ok=True)

for idx, item in enumerate(selected):
    img_dst = os.path.join(out_img_dir, f"{idx}.jpg")
    lbl_dst = os.path.join(out_lbl_dir, f"{idx}.txt")

    shutil.copy(item["img"], img_dst)

    if os.path.exists(item["lbl"]):
        shutil.copy(item["lbl"], lbl_dst)

print("✅ Dataset created successfully!")

Total images: 3712
Buckets -> High: 1351, Medium: 1275, Low: 1086
Selected images: 1000
✅ Dataset created successfully!


)

In [38]:
len(os.listdir("/content/final_dataset/images"))

1000